# Phase 10 V2: TabularGRPO Fraud Decision Research

This notebook fixes the failure observed in V1: **Approve-All policy collapse**.

V1 produced approximately:

- Fraud approval rate: 100%
- Action distribution: all transactions approved
- MoE expert utilization: balanced

That diagnosis means the representation/router was not the main failure. The decision-learning objective and training signal were.

## V2 changes

1. **Balanced training sampler** so fraud appears often enough during policy learning.
2. **Supervised warm-start** so the Transformer + MoE first learns fraud structure with weighted BCE.
3. **Structured GRPO action groups** containing all four actions plus policy samples, preventing groups containing only Approve.
4. **Multi-component rewards** so fraud protection, friction, review cost, and false declines are logged separately.
5. **Policy-entropy monitoring** to detect action collapse independently from MoE routing entropy.
6. **Zero-variance/advantage-collapse monitoring** for ineffective GRPO groups.
7. **Collapse early stopping** if one action exceeds 95% on validation for three epochs.
8. **Validation constraints** so high reward cannot win while fraud approval is unacceptable.
9. **Trivial-policy benchmarks**: Approve-All, Verify-All, Review-All, Decline-All.
10. **XGBoost-derived policy benchmark** when a Phase 2 XGBoost model is available.
11. **Multiple reward scenarios** for reward-sensitivity testing.
12. **Fraud-only inference report** over held-out fraud rows after training.

> Research POC only. Reward values and constraints are experimental assumptions, not banking policy.


## What remains unchanged

We deliberately keep the core architecture small and similar to V1:

`30 features → feature tokens → Transformer → top-2 MoE → 4-action policy`

We are fixing the learning system before adding more layers or experts. This lets us attribute improvements to sampling, warm-starting, reward design, and GRPO signal quality instead of model size.


In [ ]:
# 1. Mount Drive, clone/update project, install dependencies
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime, timezone
import os,sys,json,random,shutil,subprocess,hashlib,time,platform
import numpy as np
import pandas as pd

SEED=42
random.seed(SEED); np.random.seed(SEED)
REPO='https://github.com/deepak-malviyaa/enterprise-adaptive-learning-lab.git'
PROJECT_ROOT=Path('/content/enterprise-adaptive-learning-lab')
DRIVE_ROOT=Path('/content/drive/MyDrive/enterprise-adaptive-learning-lab-data')
DATA_PATH=DRIVE_ROOT/'raw'/'creditcard.csv'
RUNS_ROOT=DRIVE_ROOT/'outputs'/'runs'; RUNS_ROOT.mkdir(parents=True,exist_ok=True)

if (PROJECT_ROOT/'.git').exists(): r=subprocess.run(['git','-C',str(PROJECT_ROOT),'pull','--ff-only'],text=True,capture_output=True)
elif PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT); r=subprocess.run(['git','clone',REPO,str(PROJECT_ROOT)],text=True,capture_output=True)
else: r=subprocess.run(['git','clone',REPO,str(PROJECT_ROOT)],text=True,capture_output=True)
print(r.stdout); print(r.stderr)
if r.returncode!=0: raise RuntimeError('Git operation failed.')
pyproject=PROJECT_ROOT/'pyproject.toml'; t=pyproject.read_text().replace('>=3.10,<3.13','>=3.10,<3.14'); pyproject.write_text(t)
os.chdir(PROJECT_ROOT)
subprocess.run([sys.executable,'-m','pip','install','-q','-e','.'],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','torch','xgboost>=2,<4'],check=True)
if not DATA_PATH.exists(): raise FileNotFoundError(DATA_PATH)
print('Environment ready.')


In [ ]:
# 2. Dataset contract and chronological 60/20/20 split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score,roc_auc_score,precision_score,recall_score,f1_score,brier_score_loss,confusion_matrix

def sha(path):
    d=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''): d.update(b)
    return d.hexdigest()
DATASET_SHA256=sha(DATA_PATH)
df=pd.read_csv(DATA_PATH)
EXPECTED=['Time']+[f'V{i}' for i in range(1,29)]+['Amount','Class']
missing=sorted(set(EXPECTED)-set(df.columns))
if missing or set(df.Class.unique())!={0,1}: raise ValueError(f'Invalid ULB dataset: {missing}')
df=df.sort_values('Time',kind='mergesort').reset_index(drop=True); df['row_id']=np.arange(len(df))
a=int(len(df)*.60); b=int(len(df)*.80)
train_df=df.iloc[:a].copy(); val_df=df.iloc[a:b].copy(); test_df=df.iloc[b:].copy()
FEATURES=['Time']+[f'V{i}' for i in range(1,29)]+['Amount']
X_train_df=train_df[FEATURES].copy(); X_val_df=val_df[FEATURES].copy(); X_test_df=test_df[FEATURES].copy()
y_train=train_df.Class.to_numpy(np.int64); y_val=val_df.Class.to_numpy(np.int64); y_test=test_df.Class.to_numpy(np.int64)
scaler=StandardScaler().fit(X_train_df[['Time','Amount']])
for f in [X_train_df,X_val_df,X_test_df]: f.loc[:,['Time','Amount']]=scaler.transform(f[['Time','Amount']])
X_train=X_train_df.to_numpy(np.float32); X_val=X_val_df.to_numpy(np.float32); X_test=X_test_df.to_numpy(np.float32)
print('Checksum:',DATASET_SHA256); print('Fraud counts:',y_train.sum(),y_val.sum(),y_test.sum())


## Change 1: balanced policy-training batches

Natural fraud prevalence is retained in validation and test. Only the training loader uses inverse-frequency weighted sampling. This gives fraud examples enough influence during representation and policy learning without distorting final evaluation prevalence.


In [ ]:
# 3. PyTorch runtime and balanced sampler
import torch
from torch import nn
from torch.utils.data import TensorDataset,DataLoader,WeightedRandomSampler
torch.manual_seed(SEED)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
class_counts=np.bincount(y_train); class_weights=1.0/class_counts; sample_weights=class_weights[y_train]
sampler=WeightedRandomSampler(torch.tensor(sample_weights,dtype=torch.double),num_samples=len(sample_weights),replacement=True)
train_dataset=TensorDataset(torch.tensor(X_train),torch.tensor(y_train,dtype=torch.long))
BALANCED_LOADER=DataLoader(train_dataset,batch_size=512,sampler=sampler)
VAL_X=torch.tensor(X_val,dtype=torch.float32,device=DEVICE); TEST_X=torch.tensor(X_test,dtype=torch.float32,device=DEVICE)
pos_weight=torch.tensor([(y_train==0).sum()/(y_train==1).sum()],dtype=torch.float32,device=DEVICE)
print('Device:',DEVICE,'Natural train prevalence:',y_train.mean(),'Sampler is inverse-frequency balanced.')


## Architecture

The same core architectural idea is retained. This is intentional: V2 is a training-system experiment, not an architecture-size experiment.


In [ ]:
# 4. Feature tokenizer, Transformer, top-2 MoE, classification head and policy head
class Tokenizer(nn.Module):
    def __init__(self,n=30,d=48): super().__init__(); self.w=nn.Parameter(torch.randn(n,d)*.02); self.b=nn.Parameter(torch.zeros(n,d)); self.ids=nn.Parameter(torch.randn(n,d)*.02)
    def forward(self,x): return x.unsqueeze(-1)*self.w+self.b+self.ids
class Experts(nn.Module):
    def __init__(self,d=48,e=4): super().__init__(); self.router=nn.Linear(d,e); self.experts=nn.ModuleList([nn.Sequential(nn.Linear(d,96),nn.GELU(),nn.Linear(96,d)) for _ in range(e)]); self.last_probs=None
    def forward(self,x):
        p=torch.softmax(self.router(x),-1); self.last_probs=p; values,ids=p.topk(2,-1); out=torch.zeros_like(x)
        for k in range(2):
            for i,expert in enumerate(self.experts):
                mask=ids[:,k]==i
                if mask.any(): out[mask]+=values[mask,k:k+1]*expert(x[mask])
        return out
    def balance_loss(self):
        m=self.last_probs.mean(0); return ((m-1/len(m))**2).mean()
class TabularEncoder(nn.Module):
    def __init__(self):
        super().__init__(); self.tok=Tokenizer(); self.cls=nn.Parameter(torch.zeros(1,1,48)); layer=nn.TransformerEncoderLayer(48,4,96,.1,batch_first=True,norm_first=True); self.encoder=nn.TransformerEncoder(layer,2); self.moe=Experts()
    def forward(self,x):
        z=self.tok(x); z=self.encoder(torch.cat([self.cls.expand(len(x),-1,-1),z],1))[:,0]; return z+self.moe(z)
class WarmStartModel(nn.Module):
    def __init__(self): super().__init__(); self.encoder=TabularEncoder(); self.fraud_head=nn.Linear(48,1)
    def forward(self,x): return self.fraud_head(self.encoder(x)).squeeze(-1)
class PolicyModel(nn.Module):
    def __init__(self,encoder=None): super().__init__(); self.encoder=encoder if encoder is not None else TabularEncoder(); self.policy_head=nn.Linear(48,4)
    def forward(self,x): return self.policy_head(self.encoder(x))


## Change 2: supervised fraud warm-start

Before GRPO, the shared Transformer + MoE encoder learns to separate fraud from legitimate transactions using weighted BCE and balanced sampling. GRPO therefore starts from a fraud-aware representation instead of random features.


In [ ]:
# 5. Supervised warm-start using weighted BCE; select by natural-validation PR-AUC
warm=WarmStartModel().to(DEVICE); opt=torch.optim.AdamW(warm.parameters(),5e-4,weight_decay=1e-4); loss_fn=nn.BCEWithLogitsLoss(pos_weight=pos_weight)
best=-1.; best_state=None; stale=0; warm_history=[]
for epoch in range(1,21):
    warm.train(); total=0.
    for xb,yb in BALANCED_LOADER:
        xb=xb.to(DEVICE); target=yb.float().to(DEVICE); opt.zero_grad(); logits=warm(xb); loss=loss_fn(logits,target)+.05*warm.encoder.moe.balance_loss(); loss.backward(); nn.utils.clip_grad_norm_(warm.parameters(),5.); opt.step(); total+=loss.item()*len(xb)
    warm.eval();
    with torch.no_grad(): vp=torch.sigmoid(warm(VAL_X)).cpu().numpy()
    score=average_precision_score(y_val,vp); warm_history.append({'epoch':epoch,'loss':total/len(train_dataset),'val_pr_auc':score})
    if score>best: best=score; best_state={k:v.cpu().clone() for k,v in warm.state_dict().items()}; stale=0
    else: stale+=1
    if stale>=5: break
warm.load_state_dict(best_state)
print('Warm-start validation PR-AUC:',best)


## Change 3: explicit multi-component rewards

The reward is still synthetic, but every component is visible. V2 can later change each component without hiding the trade-off inside one unexplained scalar.


In [ ]:
# 6. Reward scenarios and component function
ACTIONS=['Approve','Verify','Review','Decline']
REWARD_SCENARIOS={
 'balanced':{'fraud_approve':-25.,'fraud_verify':5.,'fraud_review':7.,'fraud_decline':12.,'legit_approve':1.,'legit_verify':-2.,'legit_review':-3.,'legit_decline':-10.},
 'fraud_aggressive':{'fraud_approve':-50.,'fraud_verify':6.,'fraud_review':9.,'fraud_decline':15.,'legit_approve':1.,'legit_verify':-2.,'legit_review':-3.,'legit_decline':-10.},
 'customer_sensitive':{'fraud_approve':-20.,'fraud_verify':5.,'fraud_review':7.,'fraud_decline':10.,'legit_approve':1.,'legit_verify':-3.,'legit_review':-5.,'legit_decline':-25.},
}
SCENARIO='balanced'; cfg=REWARD_SCENARIOS[SCENARIO]
REWARD=torch.tensor([[cfg['legit_approve'],cfg['legit_verify'],cfg['legit_review'],cfg['legit_decline']],[cfg['fraud_approve'],cfg['fraud_verify'],cfg['fraud_review'],cfg['fraud_decline']]],dtype=torch.float32,device=DEVICE)
print(pd.DataFrame(REWARD.cpu().numpy(),index=['Legitimate','Fraud'],columns=ACTIONS))


## Change 4: structured action groups

V1 sampled all group actions from the policy. Once Approve dominated, groups often became homogeneous. V2 guarantees one copy of every action in each group, then adds policy-sampled actions. This provides relative reward contrast even when the policy begins to collapse.


In [ ]:
# 7. GRPO V2 settings, constraints, evaluation helpers
GROUP_SIZE=8; CLIP=.20; KL_BETA=.02; ENTROPY_BETA=.05; BALANCE_BETA=.10
MAX_FRAUD_APPROVAL=.15; MAX_FALSE_DECLINE=.03; MAX_REVIEW_VERIFY=.10

def evaluate_policy(model,X,y,reward_matrix):
    model.eval()
    with torch.no_grad(): probs=torch.softmax(model(X),-1); actions=probs.argmax(-1).cpu().numpy(); p=probs.cpu().numpy(); routing=model.encoder.moe.last_probs.cpu().numpy()
    rewards=reward_matrix.cpu().numpy()[y,actions]
    fraud_actions=actions[y==1]; legit_actions=actions[y==0]
    return {'mean_reward':float(rewards.mean()),'fraud_approval_rate':float((fraud_actions==0).mean()),'false_decline_rate':float((legit_actions==3).mean()),'intervention_rate':float(np.isin(actions,[1,2,3]).mean()),'review_verify_rate':float(np.isin(actions,[1,2]).mean()),'action_distribution':np.bincount(actions,minlength=4).tolist(),'policy_entropy':float((-p*np.log(p+1e-9)).sum(1).mean()),'expert_utilization':routing.mean(0).tolist(),'routing_entropy':float((-routing*np.log(routing+1e-9)).sum(1).mean())}


## Changes 5-8: entropy, advantage-collapse monitoring, collapse stop, and constrained model selection

A checkpoint is eligible only when fraud approval, false decline, and Verify/Review rates satisfy research constraints. The notebook monitors policy entropy separately from MoE routing entropy and stops if one action owns over 95% of natural validation decisions for three epochs.


In [ ]:
# 8. Initialize policy from supervised encoder and run GRPO V2
policy=PolicyModel(warm.encoder).to(DEVICE); reference=PolicyModel().to(DEVICE); reference.load_state_dict(policy.state_dict()); reference.eval(); optimizer=torch.optim.AdamW(policy.parameters(),2e-4)
best_state=None; best_reward=-1e9; history=[]; collapse_epochs=0
for epoch in range(1,21):
    policy.train(); zero_groups=0; groups=0; rewards_seen=[]; last_kl=last_entropy=0.
    for xb,yb in BALANCED_LOADER:
        xb=xb.to(DEVICE); yb=yb.long().to(DEVICE); batch=len(xb)
        with torch.no_grad():
            old_probs=torch.softmax(policy(xb),-1)
            # Guaranteed action diversity: [0,1,2,3] plus 4 policy samples.
            fixed=torch.arange(4,device=DEVICE).unsqueeze(0).expand(batch,-1)
            sampled=torch.distributions.Categorical(old_probs).sample((GROUP_SIZE-4,)).T
            actions=torch.cat([fixed,sampled],1)
            old_log=torch.log(old_probs.gather(1,actions)+1e-9)
            rewards=REWARD[yb.unsqueeze(1),actions]
            std=rewards.std(1,keepdim=True); zero_groups+=(std<1e-6).sum().item(); groups+=batch
            advantages=(rewards-rewards.mean(1,keepdim=True))/(std+1e-6)
        probs=torch.softmax(policy(xb),-1); logp=torch.log(probs.gather(1,actions)+1e-9); ratio=torch.exp(logp-old_log)
        surrogate=torch.minimum(ratio*advantages,torch.clamp(ratio,1-CLIP,1+CLIP)*advantages).mean()
        entropy=-(probs*torch.log(probs+1e-9)).sum(1).mean()
        with torch.no_grad(): ref_probs=torch.softmax(reference(xb),-1)
        kl=(probs*(torch.log(probs+1e-9)-torch.log(ref_probs+1e-9))).sum(1).mean()
        loss=-surrogate+KL_BETA*kl-ENTROPY_BETA*entropy+BALANCE_BETA*policy.encoder.moe.balance_loss()
        optimizer.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(policy.parameters(),5.); optimizer.step(); rewards_seen.append(rewards.mean().item()); last_kl=float(kl.item()); last_entropy=float(entropy.item())
    val=evaluate_policy(policy,VAL_X,y_val,REWARD); max_share=max(val['action_distribution'])/len(y_val); eligible=val['fraud_approval_rate']<=MAX_FRAUD_APPROVAL and val['false_decline_rate']<=MAX_FALSE_DECLINE and val['review_verify_rate']<=MAX_REVIEW_VERIFY
    history.append({'epoch':epoch,'train_group_reward':float(np.mean(rewards_seen)),'zero_variance_fraction':zero_groups/groups,'kl':last_kl,'train_action_entropy':last_entropy,'validation_mean_reward':val['mean_reward'],'validation_fraud_approval_rate':val['fraud_approval_rate'],'validation_false_decline_rate':val['false_decline_rate'],'validation_review_verify_rate':val['review_verify_rate'],'validation_policy_entropy':val['policy_entropy'],'max_action_share':max_share,'eligible':eligible})
    print('epoch',epoch,'reward',round(val['mean_reward'],4),'fraud approve',round(val['fraud_approval_rate'],4),'false decline',round(val['false_decline_rate'],4),'max action',round(max_share,4),'eligible',eligible)
    if max_share>.95: collapse_epochs+=1
    else: collapse_epochs=0
    if eligible and val['mean_reward']>best_reward: best_reward=val['mean_reward']; best_state={k:v.cpu().clone() for k,v in policy.state_dict().items()}
    if collapse_epochs>=3: print('Early stop: action-policy collapse detected.'); break
if best_state is None: print('WARNING: no checkpoint met constraints; retaining final state for diagnosis only.')
else: policy.load_state_dict(best_state); print('Loaded best constraint-eligible checkpoint.')


## Change 9: trivial-policy and learned-policy benchmarks

A sophisticated policy is useful only if it beats simple strategies under the same reward assumptions and constraints.


In [ ]:
# 9. Compare trivial policies with V2 on natural test prevalence
reward_np=REWARD.cpu().numpy()
def evaluate_actions(actions,y):
    rewards=reward_np[y,actions]; return {'mean_reward':float(rewards.mean()),'fraud_approval_rate':float((actions[y==1]==0).mean()),'false_decline_rate':float((actions[y==0]==3).mean()),'review_verify_rate':float(np.isin(actions,[1,2]).mean()),'action_distribution':np.bincount(actions,minlength=4).tolist()}
benchmark={'Approve-All':evaluate_actions(np.zeros(len(y_test),int),y_test),'Verify-All':evaluate_actions(np.ones(len(y_test),int),y_test),'Review-All':evaluate_actions(np.full(len(y_test),2),y_test),'Decline-All':evaluate_actions(np.full(len(y_test),3),y_test),'TabularGRPO-V2':evaluate_policy(policy,TEST_X,y_test,REWARD)}
display(pd.DataFrame(benchmark).T)


## Change 10: XGBoost-derived decision baseline

When a Phase 2 XGBoost model exists, validation data selects three probability cut points for Approve, Verify, Review, and Decline. This asks whether GRPO adds value over a much simpler fraud-score-based decision rule.


In [ ]:
# 10. Optional XGBoost policy benchmark
import xgboost as xgb
xgb_runs=sorted([p for p in RUNS_ROOT.glob('*_phase2_fraud_xgboost') if (p/'xgboost_model.json').exists()],reverse=True)
xgb_benchmark=None
if xgb_runs:
    model=xgb.XGBClassifier(); model.load_model(xgb_runs[0]/'xgboost_model.json')
    # XGBoost notebook used raw features, so use original chronological frames.
    raw_val=val_df[FEATURES]; raw_test=test_df[FEATURES]
    val_prob=model.predict_proba(raw_val)[:,1]; test_prob=model.predict_proba(raw_test)[:,1]
    candidates=[.01,.02,.05,.10,.20,.30,.50,.70,.80,.90]
    best=None
    for a in candidates:
        for b in candidates:
            for c in candidates:
                if not a<b<c: continue
                actions=np.select([val_prob<a,val_prob<b,val_prob<c],[0,1,2],default=3).astype(int); m=evaluate_actions(actions,y_val)
                eligible=m['fraud_approval_rate']<=MAX_FRAUD_APPROVAL and m['false_decline_rate']<=MAX_FALSE_DECLINE and m['review_verify_rate']<=MAX_REVIEW_VERIFY
                if eligible and (best is None or m['mean_reward']>best[0]): best=(m['mean_reward'],a,b,c)
    if best:
        _,a,b,c=best; actions=np.select([test_prob<a,test_prob<b,test_prob<c],[0,1,2],default=3).astype(int); xgb_benchmark=evaluate_actions(actions,y_test); xgb_benchmark['thresholds']=[a,b,c]; benchmark['XGBoost-policy']=xgb_benchmark
        print('XGBoost policy thresholds:',[a,b,c]); display(pd.DataFrame(benchmark).T)
    else: print('No XGBoost threshold combination met V2 constraints.')
else: print('No Phase 2 XGBoost model found; skipping this optional benchmark.')


## Change 11: reward sensitivity

The same trained policy is evaluated under balanced, fraud-aggressive, and customer-sensitive reward tables. This does not replace retraining under each reward scenario, but it reveals how sensitive conclusions are to the reward assumptions.


In [ ]:
# 11. Evaluate V2 actions under multiple reward scenarios
policy.eval()
with torch.no_grad(): test_probs=torch.softmax(policy(TEST_X),-1).cpu().numpy(); test_actions=test_probs.argmax(1)
sensitivity=[]
for name,c in REWARD_SCENARIOS.items():
    matrix=np.array([[c['legit_approve'],c['legit_verify'],c['legit_review'],c['legit_decline']],[c['fraud_approve'],c['fraud_verify'],c['fraud_review'],c['fraud_decline']]])
    sensitivity.append({'scenario':name,'mean_reward':float(matrix[y_test,test_actions].mean())})
reward_sensitivity=pd.DataFrame(sensitivity); display(reward_sensitivity)


## Fraud-only inference diagnostic

This section answers the practical question: **when the saved V2 policy sees held-out fraud transactions, what does it actually do and how confident is it?**


In [ ]:
# 12. Inference over every fraud row in the held-out test partition
fraud_mask=y_test==1; fraud_tensor=TEST_X[torch.tensor(fraud_mask,device=DEVICE)]
policy.eval()
with torch.no_grad(): fraud_probs=torch.softmax(policy(fraud_tensor),-1).cpu().numpy(); fraud_actions=fraud_probs.argmax(1)
fraud_rows=test_df.loc[fraud_mask,['row_id','Time','Amount','Class']].reset_index(drop=True)
fraud_inference=fraud_rows.copy(); fraud_inference['approve_probability']=fraud_probs[:,0]; fraud_inference['verify_probability']=fraud_probs[:,1]; fraud_inference['review_probability']=fraud_probs[:,2]; fraud_inference['decline_probability']=fraud_probs[:,3]; fraud_inference['selected_action_id']=fraud_actions; fraud_inference['selected_action']=[ACTIONS[i] for i in fraud_actions]; fraud_inference['policy_confidence']=fraud_probs.max(1); fraud_inference['policy_entropy']=-(fraud_probs*np.log(fraud_probs+1e-9)).sum(1)
summary=fraud_inference.selected_action.value_counts().reindex(ACTIONS,fill_value=0); display(summary.to_frame('fraud_count')); display(fraud_inference.sort_values('approve_probability',ascending=False).head(20))


## Save V2 artifacts

The V2 run is kept separate from V1 so before/after comparison is explicit.


In [ ]:
# 13. Save model, histories, inference, benchmarks, and configuration
RUN_DIR=RUNS_ROOT/(datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'_phase10_tabulargrpo_v2'); RUN_DIR.mkdir(parents=True)
torch.save(policy.state_dict(),RUN_DIR/'tabulargrpo_v2.pt'); pd.DataFrame(warm_history).to_csv(RUN_DIR/'warmstart_history.csv',index=False); pd.DataFrame(history).to_csv(RUN_DIR/'grpo_history.csv',index=False); fraud_inference.to_csv(RUN_DIR/'fraud_test_inference.csv',index=False); pd.DataFrame(benchmark).T.to_json(RUN_DIR/'benchmarks.json',orient='index',indent=2); reward_sensitivity.to_csv(RUN_DIR/'reward_sensitivity.csv',index=False)
config={'changes':['balanced_sampler','supervised_warmstart','structured_action_groups','multi_component_rewards','policy_entropy_monitoring','zero_variance_monitoring','collapse_early_stop','validation_constraints','trivial_policy_benchmarks','xgboost_policy_baseline','reward_sensitivity'],'constraints':{'max_fraud_approval':MAX_FRAUD_APPROVAL,'max_false_decline':MAX_FALSE_DECLINE,'max_review_verify':MAX_REVIEW_VERIFY},'reward_scenario':SCENARIO,'dataset_sha256':DATASET_SHA256,'seed':SEED}
(RUN_DIR/'config.json').write_text(json.dumps(config,indent=2),encoding='utf-8')
print('Saved:',RUN_DIR)


## V2 acceptance rule

V2 is not accepted merely because mean reward increases. It should:

- Beat Approve-All on meaningful decision behavior
- Reduce fraud approval dramatically from V1's 100%
- Keep false declines within the research constraint
- Keep Verify/Review usage within the research constraint
- Avoid one-action collapse
- Maintain useful policy entropy and low advantage-collapse fraction
- Remain competitive with the XGBoost-derived policy

If no checkpoint satisfies these constraints, the correct conclusion is **V2 is still not ready**, not to hide the result.
